# Agno Agents Implementation

Source: 
- Github repo: https://github.com/agno-agi/agno
- Tutorial summary: https://x.com/ashpreetbedi/status/1924193924995744158?t=gzOZLk2Rc2baSbsVfk5_Eg&s=19

> Note: Running the file in a .py file is more recommended since the progress bar is printed one by one in the .ipynb format.

In [6]:
# Setup environment variables and models
import os
from dotenv import load_dotenv
from agno.models.anthropic import Claude
from agno.models.openai import OpenAIChat

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

openai_model = OpenAIChat(id="gpt-4.1-mini", api_key=openai_api_key)
claude_model = Claude(id="claude-sonnet-4-5", api_key=anthropic_api_key)

## Quicktests

In [ ]:
# Quick test on creating an agent
from agno.agent import Agent
from agno.tools.hackernews import HackerNewsTools

agent = Agent(
    model=claude_model,
    tools=[HackerNewsTools()],
    markdown=True,
)
agent.print_response("Write a report on trending startups and products.", stream=True)

Output()

In [ ]:
# Test searching with other search tool
from agno.agent import Agent
from agno.tools.tavily import TavilyTools

agent = Agent(model=openai_model, tools=[TavilyTools(api_key=tavily_api_key)], markdown=True)
agent.print_response("Search tavily for the latest news about NVIDIA activities", stream=True)

Output()

In [11]:
from agno.agent import Agent
from agno.tools.tavily import TavilyTools
from agno.tools.yfinance import YFinanceTools

# Multi-agent team example
web_agent = Agent(
    name="Web Search Agent",
    role="Search the web for information",
    model=openai_model,
    tools=[TavilyTools(api_key=tavily_api_key)],
    instructions="Always include sources",
    markdown=True,
)

finance_agent = Agent(
    name="Finance Agent",
    role="Get financial data",
    model=openai_model,
    tools=[YFinanceTools(stock_price=True, analyst_recommendations=True)],
    instructions="Use tables to display data",
    markdown=True,
)

# Create a team with a leader agent
agent_team = Agent(
    team=[web_agent, finance_agent],
    model=openai_model,
    instructions=["Always include sources", "Use tables to display data"],
    markdown=True,
)

# Use the team
agent_team.print_response("What's the current stock price of NVDA and recent news about it?", stream=True)

Output()

In [ ]:
from agno.agent import Agent
from agno.tools.tavily import TavilyTools
from agno.tools.yfinance import YFinanceTools

# Multi-agent team example
web_agent = Agent(
    name="Web Search Agent",
    role="Search the web for information",
    model=openai_model,
    tools=[TavilyTools(api_key=tavily_api_key)],
    # instructions="Always include sources",
    markdown=True,
)

finance_agent = Agent(
    name="Finance Agent",
    role="Get financial data",
    model=openai_model,
    tools=[YFinanceTools(stock_price=True, analyst_recommendations=True)],
    # instructions="Use tables to display data",
    markdown=True,
)

# Create a team with a leader agent
agent_team = Agent(
    team=[web_agent, finance_agent],
    model=openai_model,
    # instructions=["Always include sources", "Use tables to display data"],
    markdown=True,
)

# Use the team
agent_team.print_response("What's the current stock price of NVDA and recent news about it?", stream=True)

Output()

___

## Level 1: Agent with tools and instructions. 

When people say agents are just LLM + tool calls in a loop, this is what they mean (this also tells you their level of understanding). Instructions "teach" the Agent how to achieve its task and tools let Agents interact with external environments to push or pull data. Here's an Agent that helps developers build Agents using Agno (so clean 🤩)

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools

# Initialize the Agent with instructions, tools, and a model
agno_assist = Agent(
    name="Agno Assistant",
    model=OpenAIChat(id="gpt-4.1", api_key=openai_api_key),
    description="""
    You are Agno AGI, an autonomous agent that can build agents using the Agno framework Your goal is to help developers understand and use Agno by providing
    explanations, working code examples, and optional visual and audio explanations of key concepts.
    """,
    instructions="Search the web for information about Agno",
    tools=[DuckDuckGoTools()],
    add_datetime_to_instructions=True,
    markdown=True
)

# Run the Agent
agno_assist.print_response("What is Agno?", stream=True)

## Level 2: Agent with knowledge and storage. 

Rarely does a model have all the information it needs to achieve its task and we obviously can't jam everything in the context, so we give the Agent knowledge that it searches at runtime (i.e Agentic RAG or Dynamic few-shot). 

Knowledge search needs to be hybrid (full-text and semantic). Hybrid search + reranking is the best out-of-the-box Agentic Search strategy you can use.

Storage saves the Agent's state in a database. LLM calls are "stateless" and storage makes Agents "stateful" by storing messages in a database and adding them to the current call as needed. 

If you're using chatgpt, storage is what lets us continue the chat after closing the tab and each chat thread that you see on the left navbar is a "session" in storage. 

Storage also saves the session state (very useful) but that's for another day. Here's how knowledge & storage look like in code: 

In [ ]:
import os
from dotenv import load_dotenv
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools

